#### Investigating the outliers in rf1, rf2, and sf2 datasets

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
import numpy as np
import matplotlib.pyplot as plt
from moc.metrics.distribution_metrics import pce, multivariate_energy_score, mse
import torch

In [6]:
import wandb
wandb.login(key="9d338bfb8d6dd9ab97384ee89b11f332ae3e12b8") #ELNURA'S KEY

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: ryuzaki to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#### Investigating dataset dimensionality

In [3]:
datasets = [
            ['camehl', 'households'], ['cevid', 'air'], 
            ['cevid', 'births1'],
            ['cevid', 'births2'], 
            ['cevid', 'wage'], 
            ['mulan', 'scm20d'],
            ['mulan', 'rf2'], ['mulan', 'rf1'], ['mulan', 'sf2'],
            ['mulan', 'scm1d'], ['mulan', 'wq'], ['mulan', 'scpf'], 
            ['feldman', 'meps_21'], ['feldman', 'meps_19'],['feldman', 'meps_20'],
            ['feldman', 'house'], ['feldman', 'bio'], 
            ['feldman', 'blog_data'],
            ['del_barrio', 'calcofi'],
            ['del_barrio', 'ansur2'], ['wang', 'taxi']
            ]

In [4]:
config = get_config()
config.device = 'cuda'

In [8]:
for data_group, data_name in datasets:
    rc = RunConfig(config, data_group, data_name)
    datamodule = RealDataModule(rc, num_workers=8)
    p, q = datamodule.input_dim, datamodule.output_dim 
    print(f"Dataset: {data_name} with size {len(datamodule.data_train)} Input Dim: {p}, Output Dim: {q}, ")

Dataset: households with size 2920 Input Dim: 14, Output Dim: 4, 
Dataset: air with size 4317 Input Dim: 15, Output Dim: 6, 
Dataset: births1 with size 4317 Input Dim: 23, Output Dim: 2, 
Dataset: births2 with size 4317 Input Dim: 24, Output Dim: 4, 
Dataset: wage with size 4317 Input Dim: 78, Output Dim: 2, 
Dataset: scm20d with size 3800 Input Dim: 60, Output Dim: 16, 
Dataset: rf2 with size 3818 Input Dim: 64, Output Dim: 8, 
Dataset: rf1 with size 3818 Input Dim: 64, Output Dim: 8, 
Dataset: sf2 with size 426 Input Dim: 31, Output Dim: 3, 
Dataset: scm1d with size 4218 Input Dim: 279, Output Dim: 16, 
Dataset: wq with size 424 Input Dim: 16, Output Dim: 14, 
Dataset: scpf with size 454 Input Dim: 8, Output Dim: 3, 
Dataset: meps_21 with size 7145 Input Dim: 138, Output Dim: 2, 
Dataset: meps_19 with size 7209 Input Dim: 138, Output Dim: 2, 
Dataset: meps_20 with size 8087 Input Dim: 138, Output Dim: 2, 
Outlier present!
Dataset: house with size 10123 Input Dim: 17, Output Dim: 2, 


In [4]:
config = get_config()
config.device = 'cuda'
seeds = [0, 42, 866, 12, 4]
data_group, data_name = 'mulan', 'rf1'

In [7]:
hparams = {
            'model': 'mixture',
            'lambda': 0.0,
            'prerank': 'none'
        }
rc = RunConfig(config, data_group, data_name, hparams = hparams, seed = 866)
datamodule = RealDataModule(rc, num_workers=8, seed = 866)
p, q = datamodule.input_dim, datamodule.output_dim #268,16
model = MixtureLightningModule(p, q)
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
wandb.finish()


/mnt/default/elnura_workspace/Multivariate-recalibration/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/default/elnura_workspace/Multivariate-recalibra ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/mnt/default/elnura_workspace/Multivariate-recalibration/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/2025-07-20/21-25-26/mulan/rf1/model=mixture,lambda=0.0,prerank=none/0/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Checking False, False


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/marg_val,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/nll,█▇▇▆▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁
train/prerank_val,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total_loss,█▇▇▆▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
val/nll,█▇▇▄▃▁▅▁▁▁▂▂▂▁▄▃▄▄▄▄▆▆▆▅
epoch,23
train/marg_val,0
train/nll,0.56402
train/prerank_val,0


In [8]:
ckpt_path = trainer.checkpoint_callback.best_model_path
best_model = MixtureLightningModule.load_from_checkpoint(ckpt_path)
best_model.eval().to(config.device)
test_loader = datamodule.test_dataloader()

for (x, y, idx) in test_loader:
    # if i!=6: continue
    x = x.to(config.device)
    y = y.to(config.device)
    dist = best_model.predict(x)
    nll_value = -dist.log_prob(y)
    print(nll_value.mean().item())
    # for j, nll in zip(idx, nll_value):
    #     print(j.item(), nll.item())

6.034688949584961
5.920483112335205
1777.9873046875
6.015256881713867
6.131124019622803
6.061954975128174
6.11771821975708
6.1075639724731445


#### Justifying PCA: reducing dimesnionality in scm20d and scm1d and showing that we get either the same or better PCE

In [3]:
config = get_config()
config.device = 'cuda'
data_group, data_name = 'mulan', 'scm20d'
# preranks = ['mean', 'variance', 'dependency', 'density', 'cdf']
# lambdas = [[10.0, 5.0, 10.0, 5.0, 10.0], [10.0, 10.0, 10.0, 10.0, 10.0]]

In [ ]:
hparams = {
            'model': 'mixture',
            'lambda': 10.0,
            'prerank': 'mean',
            'double_reg_type': 'pca'
        }
rc = RunConfig(config, data_group, data_name, hparams = hparams)
datamodule = RealDataModule(rc, num_workers=8)
p, q = datamodule.input_dim, datamodule.output_dim #268,16
model = MixtureLightningModule(p, q, lambda_reg=hparams['lambda'], reg_type = 'pce-kde', 
                            prerank=hparams['prerank'], double_reg = True, double_reg_type = hparams['double_reg_type'])
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
# wandb.finish()